# 01 — Bayesian Foundations
**References:** Gelman et al. BDA3 Ch. 1 · Bernardo & Smith (1994) Ch. 1–2 · McElreath (2020) Ch. 2

## Narrative thread
```
Two philosophies -> Coherence -> Bayes theorem for inference -> The three components -> Posterior summaries
```

## 1. Two philosophies of probability

| Aspect | Frequentist | Bayesian |
|---|---|---|
| What is probability? | Long-run relative frequency | Degree of belief (subjective or objective) |
| Parameters $\theta$ | Fixed, unknown constants | Random variables — have distributions |
| Data $\mathbf{x}$ | Random (varies across experiments) | Fixed — what we actually observed |
| Inference on $\theta$ | Confidence intervals, p-values | Posterior distribution $p(\theta\mid\mathbf{x})$ |
| Prior knowledge | Excluded by design | Explicitly encoded in $p(\theta)$ |
| Prediction | Plug-in MLE | Posterior predictive $p(\tilde x \mid \mathbf{x})$ |

**de Finetti's theorem:** Any exchangeable sequence of binary random variables can be represented as a mixture of i.i.d. Bernoulli sequences — with the mixing distribution playing the role of the prior. This provides a foundational justification for Bayesian inference without requiring a metaphysical interpretation of probability.

**Coherence (Dutch Book argument):** An agent whose beliefs violate the probability axioms can be made to accept a combination of bets that guarantees a loss. Bayesian inference is the *unique coherent* way to update beliefs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── The three components: prior, likelihood, posterior ────────────────────
# Model: X1,...,Xn ~ N(theta, 1),  theta ~ N(mu0, tau0^2)
# Posterior: theta | X ~ N(mu_n, tau_n^2)
#   tau_n^{-2} = tau0^{-2} + n
#   mu_n = (mu0/tau0^2 + n*xbar) / (1/tau0^2 + n)

mu0   = 0.0    # prior mean
tau0  = 2.0    # prior SD
sigma = 1.0    # known likelihood SD

# Two datasets: one supporting prior, one contradicting it
data_A = np.array([0.3, -0.5, 0.8, 0.1, -0.2])  # near prior center
data_B = np.array([4.1, 3.8, 5.2, 4.6, 3.9])    # far from prior

theta_grid = np.linspace(-4, 8, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, label in zip(axes, [data_A, data_B], ['Data near prior', 'Data far from prior']):
    n    = len(data)
    xbar = data.mean()

    # Posterior parameters
    tau_n_inv2 = 1/tau0**2 + n/sigma**2
    mu_n       = (mu0/tau0**2 + n*xbar/sigma**2) / tau_n_inv2
    tau_n      = 1/np.sqrt(tau_n_inv2)

    prior     = stats.norm.pdf(theta_grid, mu0, tau0)
    posterior = stats.norm.pdf(theta_grid, mu_n, tau_n)
    # Likelihood: proportional to N(theta; xbar, sigma^2/n)
    likelihood = stats.norm.pdf(theta_grid, xbar, sigma/np.sqrt(n))
    likelihood = likelihood / likelihood.max() * max(prior.max(), posterior.max())

    ax.plot(theta_grid, prior,      color='#06d6a0', lw=2.5, label=f'Prior $\\mathcal{{N}}({mu0},{tau0}^2)$')
    ax.plot(theta_grid, likelihood, color='gray',    lw=2,   linestyle='--', label=f'Likelihood (scaled), $\\bar x={xbar:.1f}$')
    ax.plot(theta_grid, posterior,  color='#4361ee', lw=2.5, label=f'Posterior $\\mathcal{{N}}({mu_n:.2f},{tau_n:.2f}^2)$')
    ax.axvline(mu_n, color='#4361ee', lw=1.5, linestyle=':', alpha=0.7)
    ax.axvline(xbar, color='gray',    lw=1.5, linestyle=':', alpha=0.7)
    ax.set_xlabel('$\\theta$'); ax.set_ylabel('Density')
    ax.set_title(f'{label}\nPosterior mean = precision-weighted average of prior and $\\bar x$')
    ax.legend(fontsize=8)

plt.suptitle('Prior + Likelihood → Posterior: Normal-Normal conjugate model', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The posterior as a complete inference summary

All Bayesian inference flows from the posterior $p(\theta \mid \mathbf{x})$:

$$p(\theta \mid \mathbf{x}) \propto \underbrace{p(\mathbf{x} \mid \theta)}_{\text{likelihood}} \cdot \underbrace{p(\theta)}_{\text{prior}}$$

**Point estimates:**
- **Posterior mean** $E[\theta \mid \mathbf{x}]$ — optimal under squared error loss
- **Posterior median** — optimal under absolute error loss
- **MAP** $\arg\max_\theta p(\theta\mid\mathbf{x})$ — optimal under 0-1 loss; coincides with MLE when prior is flat

**Interval summary — Credible interval:**
$$P(a \leq \theta \leq b \mid \mathbf{x}) = 1-\alpha$$

The **Highest Posterior Density (HPD)** interval is the shortest such interval:
$$\text{HPD}_{1-\alpha} = \{\theta : p(\theta \mid \mathbf{x}) \geq h_\alpha\}$$
where $h_\alpha$ is chosen so the region has probability $1-\alpha$.

**Posterior predictive distribution:** For a new observation $\tilde{x}$:
$$p(\tilde{x} \mid \mathbf{x}) = \int p(\tilde{x} \mid \theta)\, p(\theta \mid \mathbf{x})\,d\theta$$

> This **averages predictions over all parameter values**, weighted by their posterior probability — automatic propagation of parameter uncertainty.

In [ ]:
# ── Posterior predictive distribution ─────────────────────────────────────
# Normal-Normal: posterior predictive is N(mu_n, tau_n^2 + sigma^2)

data = np.array([2.1, 1.8, 2.4, 2.0, 2.3, 1.9])
n    = len(data); xbar = data.mean()
mu0, tau0, sigma = 0.0, 3.0, 1.0

tau_n_inv2 = 1/tau0**2 + n/sigma**2
mu_n       = (mu0/tau0**2 + n*xbar/sigma**2) / tau_n_inv2
tau_n      = 1/np.sqrt(tau_n_inv2)

# Posterior predictive: N(mu_n, tau_n^2 + sigma^2)
pred_mean = mu_n
pred_sd   = np.sqrt(tau_n**2 + sigma**2)

# Plug-in predictive (ignores parameter uncertainty): N(xbar, sigma^2)
plugin_sd = sigma

x_range = np.linspace(-4, 8, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_range, stats.norm.pdf(x_range, mu_n, tau_n),
        color='#06d6a0', lw=2.5, label=f'Posterior $\\mathcal{{N}}({mu_n:.2f},{tau_n:.2f}^2)$')
ax.plot(x_range, stats.norm.pdf(x_range, pred_mean, pred_sd),
        color='#4361ee', lw=2.5, label=f'Post. predictive $\\mathcal{{N}}({pred_mean:.2f},{pred_sd:.2f}^2)$')
ax.plot(x_range, stats.norm.pdf(x_range, xbar, plugin_sd),
        color='#f72585', lw=2, linestyle='--', label=f'Plug-in $\\mathcal{{N}}(\\bar x={xbar:.2f},{plugin_sd}^2)$')
ax.set_xlabel('$\\tilde x$'); ax.set_ylabel('Density')
ax.set_title('Posterior vs Posterior Predictive\nPredictive is wider: includes parameter uncertainty')
ax.legend(fontsize=8)

# ── Sample from posterior predictive via Monte Carlo ───────────────────────
n_draws = 10_000
theta_draws = np.random.normal(mu_n, tau_n, n_draws)
x_pred_MC   = np.random.normal(theta_draws, sigma)

ax2 = axes[1]
ax2.hist(x_pred_MC, bins=60, density=True, color='#4361ee', alpha=0.7, edgecolor='white',
         label='MC posterior predictive')
ax2.plot(x_range, stats.norm.pdf(x_range, pred_mean, pred_sd), 'r-', lw=2.5, label='Analytical')
ax2.set_xlabel('$\\tilde x$'); ax2.set_ylabel('Density')
ax2.set_title('Posterior predictive via Monte Carlo:\n'
               '1. Sample $\\theta^* \\sim p(\\theta\\mid\\mathbf{x})$\n'
               '2. Sample $\\tilde x \\sim p(\\tilde x \\mid \\theta^*)$')
ax2.legend(fontsize=8)

plt.suptitle('Posterior Predictive Distribution — Propagating Parameter Uncertainty', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Bayesian updating is sequential

The posterior after seeing $\mathbf{x}_1$ becomes the prior for $\mathbf{x}_2$:
$$p(\theta \mid \mathbf{x}_1, \mathbf{x}_2) \propto p(\mathbf{x}_2 \mid \theta)\, p(\theta \mid \mathbf{x}_1) \propto p(\mathbf{x}_1, \mathbf{x}_2 \mid \theta)\, p(\theta)$$

The final result is the same whether we process data sequentially or all at once — **order invariance**.

## 4. Key takeaways

| Concept | Key idea |
|---|---|
| Prior | Encodes beliefs before seeing data |
| Likelihood | How probable is the data under each $\theta$ |
| Posterior | Updated beliefs — complete inference summary |
| Posterior predictive | Future data distribution — averages over $\theta$ |
| Credible interval | Direct probability statement about $\theta$ |
| Sequential updating | Posterior today is prior tomorrow |

**Next:** notebook 02 — conjugate models in depth: all exponential family conjugates.